# Ejemplos de Dash

Notebook con ejemplos progresivos para aprender **Dash**, el framework de Python para construir dashboards web interactivos.

**Requisitos:** `pip install dash pandas plotly`

> Nota: en Jupyter, las apps de Dash se corren con `app.run(jupyter_mode="inline")` para que se muestren dentro del notebook, en vez de abrir una pestaña del navegador aparte. Si preferís abrir la app en una pestaña nueva, usá `jupyter_mode="external"` y abrí el link que aparece en la consola.


In [ ]:
# Instalación (descomentar si hace falta)
# !pip install dash pandas plotly


## 1. App mínima de Dash

La estructura básica de toda app de Dash tiene 2 partes:
1. **`app.layout`**: define qué se ve en pantalla (los componentes).
2. **`app.run()`**: levanta el servidor.


In [8]:
from dash import Dash, html

app = Dash(__name__)

app.layout = html.Div([
    html.H1('Mi primera app de Dash'),
    html.P('Si ves este texto, la app está funcionando correctamente.')
])

app.run(debug=True, jupyter_mode='inline', port=8050)


## 2. Layout con un gráfico (dcc.Graph)

`dcc` (Dash Core Components) trae componentes interactivos: gráficos, dropdowns, sliders, inputs de texto, etc.
Acá insertamos una figura de Plotly Express dentro de un `dcc.Graph`.


In [9]:
from dash import Dash, html, dcc
import plotly.express as px

app = Dash(__name__)

tips = px.data.tips()
fig = px.scatter(tips, x='total_bill', y='tip', color='sex',
                  title='Propina vs Cuenta Total')

app.layout = html.Div([
    html.H1('Dashboard de Propinas'),
    dcc.Graph(id='grafico-propinas', figure=fig)
])

app.run(debug=True, jupyter_mode='inline', port=8051)


## 3. Callback básico: un Input, un Output

Acá está el concepto central de Dash: el **callback**.

- `Input('dropdown-dia', 'value')`: escucha cambios en el `value` del dropdown.
- `Output('grafico', 'figure')`: cada vez que el Input cambia, la función decorada se ejecuta y actualiza el `figure` del gráfico.

Es literalmente **"vincular las entradas del usuario a las actualizaciones de la visualización"**.


In [10]:
from dash import Dash, html, dcc, Input, Output
import plotly.express as px

app = Dash(__name__)
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Propinas por Día'),
    dcc.Dropdown(
        id='dropdown-dia',
        options=[{'label': dia, 'value': dia} for dia in tips['day'].unique()],
        value='Sun'
    ),
    dcc.Graph(id='grafico-dia')
])

@app.callback(
    Output('grafico-dia', 'figure'),
    Input('dropdown-dia', 'value')
)
def actualizar_grafico(dia_seleccionado):
    filtrado = tips[tips['day'] == dia_seleccionado]
    fig = px.scatter(filtrado, x='total_bill', y='tip', color='sex',
                      title=f'Propinas - {dia_seleccionado}')
    return fig

app.run(debug=True, jupyter_mode='inline', port=8052)


## 4. Múltiples Inputs → un Output

Un callback puede escuchar **varios** inputs a la vez. Acá combinamos un dropdown y un slider para filtrar los datos.


In [11]:
from dash import Dash, html, dcc, Input, Output
import plotly.express as px

app = Dash(__name__)
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Filtro combinado: Día + Monto mínimo'),

    html.Label('Día:'),
    dcc.Dropdown(
        id='dropdown-dia-2',
        options=[{'label': dia, 'value': dia} for dia in tips['day'].unique()],
        value='Sun'
    ),

    html.Label('Cuenta mínima ($):'),
    dcc.Slider(
        id='slider-monto',
        min=0, max=50, step=5, value=0,
        marks={i: f'${i}' for i in range(0, 51, 10)}
    ),

    dcc.Graph(id='grafico-combinado')
])

@app.callback(
    Output('grafico-combinado', 'figure'),
    Input('dropdown-dia-2', 'value'),
    Input('slider-monto', 'value')
)
def actualizar_combinado(dia, monto_min):
    filtrado = tips[(tips['day'] == dia) & (tips['total_bill'] >= monto_min)]
    fig = px.bar(filtrado, x='sex', y='tip', color='smoker', barmode='group',
                 title=f'{dia} | Cuenta >= ${monto_min}')
    return fig

app.run(debug=True, jupyter_mode='inline', port=8053)


## 5. Un Input → múltiples Outputs

También podés actualizar **varios** componentes al mismo tiempo desde un solo Input. Acá un dropdown actualiza tanto un gráfico como un texto con estadísticas.


In [12]:
from dash import Dash, html, dcc, Input, Output
import plotly.express as px

app = Dash(__name__)
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Estadísticas por Día'),
    dcc.Dropdown(
        id='dropdown-dia-3',
        options=[{'label': dia, 'value': dia} for dia in tips['day'].unique()],
        value='Sun'
    ),
    html.Div(id='texto-estadisticas', style={'margin': '20px 0', 'fontSize': '18px'}),
    dcc.Graph(id='grafico-3')
])

@app.callback(
    Output('grafico-3', 'figure'),
    Output('texto-estadisticas', 'children'),
    Input('dropdown-dia-3', 'value')
)
def actualizar_todo(dia):
    filtrado = tips[tips['day'] == dia]
    fig = px.histogram(filtrado, x='tip', nbins=10, title=f'Distribución de propinas - {dia}')
    promedio = filtrado['tip'].mean()
    texto = f'Propina promedio el {dia}: ${promedio:.2f} | Registros: {len(filtrado)}'
    return fig, texto

app.run(debug=True, jupyter_mode='inline', port=8054)


## 6. Callback encadenado con `State` (no dispara el callback)

`State` permite leer el valor de un componente **sin** que dispare el callback por sí solo. Es útil para formularios donde querés que la actualización ocurra recién al hacer click en un botón, no en cada tecla.


In [6]:
from dash import Dash, html, dcc, Input, Output, State
import plotly.express as px

app = Dash(__name__)
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Filtro con botón (usando State)'),
    dcc.Input(id='input-monto', type='number', placeholder='Monto mínimo', value=10),
    html.Button('Aplicar filtro', id='boton-aplicar', n_clicks=0),
    dcc.Graph(id='grafico-4')
])

@app.callback(
    Output('grafico-4', 'figure'),
    Input('boton-aplicar', 'n_clicks'),
    State('input-monto', 'value')
)
def actualizar_con_boton(n_clicks, monto_min):
    monto_min = monto_min or 0
    filtrado = tips[tips['total_bill'] >= monto_min]
    fig = px.scatter(filtrado, x='total_bill', y='tip', color='day',
                      title=f'Cuentas >= ${monto_min} (click #{n_clicks})')
    return fig

app.run(debug=True, jupyter_mode='inline', port=8055)


## 7. Dashboard combinado (ejemplo final)

Uniendo varios conceptos: dropdown, slider, checklist y dos gráficos que se actualizan juntos.


In [7]:
from dash import Dash, html, dcc, Input, Output
import plotly.express as px

app = Dash(__name__)
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Dashboard Final: Análisis de Propinas'),

    html.Div([
        html.Div([
            html.Label('Día(s):'),
            dcc.Checklist(
                id='checklist-dias',
                options=[{'label': d, 'value': d} for d in tips['day'].unique()],
                value=list(tips['day'].unique()),
                inline=True
            ),
        ], style={'marginBottom': '15px'}),

        html.Div([
            html.Label('Rango de cuenta total:'),
            dcc.RangeSlider(
                id='rangeslider-monto',
                min=int(tips['total_bill'].min()),
                max=int(tips['total_bill'].max()) + 1,
                value=[0, 50],
                marks=None,
                tooltip={'placement': 'bottom', 'always_visible': True}
            ),
        ], style={'marginBottom': '25px'}),
    ]),

    html.Div([
        dcc.Graph(id='grafico-izq', style={'width': '49%', 'display': 'inline-block'}),
        dcc.Graph(id='grafico-der', style={'width': '49%', 'display': 'inline-block'}),
    ])
])

@app.callback(
    Output('grafico-izq', 'figure'),
    Output('grafico-der', 'figure'),
    Input('checklist-dias', 'value'),
    Input('rangeslider-monto', 'value')
)
def actualizar_dashboard(dias_seleccionados, rango_monto):
    filtrado = tips[
        (tips['day'].isin(dias_seleccionados)) &
        (tips['total_bill'] >= rango_monto[0]) &
        (tips['total_bill'] <= rango_monto[1])
    ]

    fig_izq = px.scatter(filtrado, x='total_bill', y='tip', color='day',
                          title='Cuenta vs Propina')
    fig_der = px.box(filtrado, x='day', y='tip', color='sex',
                      title='Distribución de Propinas por Día y Sexo')

    return fig_izq, fig_der

app.run(debug=True, jupyter_mode='inline', port=8056)


## Notas finales

- Cada ejemplo usa un **puerto distinto** (`8050`, `8051`, etc.) para que puedas correr varias celdas sin conflicto. Si te da error de "puerto en uso", cambiá el número de `port`.
- `debug=True` activa hot-reload y muestra errores detallados en la app — muy útil mientras desarrollás.
- Si `jupyter_mode='inline'` no renderiza bien en tu entorno, probá `jupyter_mode='external'` (te va a tirar un link para abrir en el navegador) o `jupyter_mode='tab'`.
- Para desplegar la app fuera de Jupyter, guardá el código en un archivo `.py` y corré `app.run(debug=True)` normalmente (sin `jupyter_mode`), luego ejecutá `python app.py` desde la terminal.
